<a href="https://www.kaggle.com/code/airzip/mistral-suitableexperimentingkaggle?scriptVersionId=221751566" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
!pip install -q -U bitsandbytes

In [2]:
import torch
from transformers import BitsAndBytesConfig

#8-bit Quantization settings
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=7.0,
    llm_int8_has_fp16_weight=False
)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "/kaggle/input/mistral-small-24b/transformers/mistral-small-24b-instruct-2501/1"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    #torch_dtype="auto",
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model

Loading checkpoint shards:   0%|          | 0/10 [00:00<?, ?it/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(131072, 5120)
    (layers): ModuleList(
      (0-39): 40 x MistralDecoderLayer(
        (self_attn): MistralSdpaAttention(
          (q_proj): Linear8bitLt(in_features=5120, out_features=4096, bias=False)
          (k_proj): Linear8bitLt(in_features=5120, out_features=1024, bias=False)
          (v_proj): Linear8bitLt(in_features=5120, out_features=1024, bias=False)
          (o_proj): Linear8bitLt(in_features=4096, out_features=5120, bias=False)
          (rotary_emb): MistralRotaryEmbedding()
        )
        (mlp): MistralMLP(
          (gate_proj): Linear8bitLt(in_features=5120, out_features=32768, bias=False)
          (up_proj): Linear8bitLt(in_features=5120, out_features=32768, bias=False)
          (down_proj): Linear8bitLt(in_features=32768, out_features=5120, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): MistralRMSNorm((5120,), eps=1e-05)
        (post_attention_layernorm

In [4]:
prompt = "9.9和9.11哪个大？"
messages = [
    {"role": "system", "content": "扮演智能助手"},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages, 
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [5]:
generated_ids = model.generate(
    **model_inputs,
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=512
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

from IPython.display import Markdown
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
Markdown(response)

/usr/local/lib/python3.10/dist-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


9.9 和 9.11 之间，9.11 更大。

这是因为在小数比较中，从左到右逐位比较，第一个不同的数字决定了大小。在 9.9 和 9.11 中，整数部分相同，小数部分的第一个数字是 9 和 1，显然 1 大于 9，所以 9.11 大于 9.9。